# Optimisations MODÈLE COMPLET — R50, FCOS R50, EfficientDet D4

Toutes les optimisations appliquées au **modèle entier** (sans découpage en zones),
en individuel **et** combinées avec l'autocast FP16. Certaines vont sous-performer
ou échouer (le NMS dynamique casse compile/cudagraphs/TRT) — **c'est voulu** : on
veut les logs pour décider la suite (optimisations par zones, etc.).

**Variantes** (`FULL_VARIANTS`)

| Variante | = | MAP@640 |
|---|---|:---:|
| `baseline` | FP32 | ✓ |
| `fp16` | autocast FP16 | ✓ |
| `torchscript` | TorchScript | ✓ |
| `compile` | torch.compile (inductor) | ✗ (=baseline) |
| `cudagraphs` | CUDA graphs | ✗ (=baseline) |
| `trt_fp16` | TensorRT FP16 | ✓ |
| `compile_fp16` | autocast + compile | ✓ |
| `cudagraphs_fp16` | autocast + cudagraphs | ✓ |
| `torchscript_fp16` | autocast + torchscript | ✓ |

**Éval** : toujours à la **taille d'optimisation (640×640)** via `preprocess`/`postprocess`
du modèle (MAP@640) — on accepte la perte de précision absolue, on veut le **delta**.
Modèles compilés/TRT → MAP à **batch=1** (shape figée).

**Robustesse** : chaque variante isolée (try/except) + **reset d'état** entre variantes
(évite la pollution cudagraph/inductor). Tout est **sauvegardé** sous le préfixe Drive.

## 0. Installation (Colab)
```bash
!pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet
```

In [ ]:
# Colab — décommenter à la première exécution
# !pip install torch-tensorrt onnx onnxruntime-gpu onnxsim effdet pycocotools --quiet

## 1. Préfixe de sortie (Drive)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from optimizations.paths import set_prefix, describe
try:
    from google.colab import drive
    drive.mount("/content/drive")
    set_prefix("/content/drive/MyDrive/ProjectDSAI2026_2")
except Exception:
    set_prefix("")          # local : chemins relatifs au projet
print(describe())

## 2. Imports & configuration

In [ ]:
from datetime import datetime
import torch, pandas as pd
import optimizations as opt
from optimizations import OptimizationRunner, RunConfig, ModelSpec, FULL_VARIANTS
from utils.data_loader import load_profiling_data, load_eval_data

CAPS = opt.print_report()

# ── Paramètres (démarrer petit pour avoir vite les logs, puis monter) ─────────
N_WARMUP   = 50
N_MEASURE  = 500       # itérations bench (1000 pour le run final)
N_PROFILE  = 150
N_EVAL     = 500       # images MAP@640 (2000 pour le run final)
DO_PROFILE = True
DO_INT8    = False
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

IMG_DIR  = "datasets/coco/val2017"
ANN_FILE = "datasets/coco/annotations/instances_val2017.json"
if DEVICE == "cpu":
    print("\n⚠ Pas de GPU CUDA — le benchmark en a besoin (torch +cuXXX).")

## 3. Données

In [ ]:
from pycocotools.coco import COCO
profile_data = load_profiling_data(IMG_DIR, ANN_FILE, n=max(2000, N_WARMUP+N_MEASURE))
eval_data, _ = load_eval_data(IMG_DIR, ANN_FILE, n=N_EVAL)
coco_gt      = COCO(ANN_FILE)
print(f"bench/profil : {len(profile_data)} | eval : {len(eval_data)} images")

## 4. Modèles et variantes

3 modèles : RetinaNet R50, FCOS R50 (torchvision), EfficientDet D4 (effdet).
`compile`/`compile_fp16` sont retirés d'effdet par défaut (inductor très lent sur
le BiFPN) — `EFFDET_COMPILE=True` pour les inclure.

In [ ]:
import models.retinanet_r50  as r50
import models.fcos_r50        as fcos
import models.efficientdet_d4 as d4

SPECS = {
    "retinanet_r50": ModelSpec("retinanet_r50", r50,  "torchvision", has_map=True),
    "fcos_r50":      ModelSpec("fcos_r50",      fcos, "torchvision", has_map=True),
    "efficientdet_d4": ModelSpec("efficientdet_d4", d4, "effdet",    has_map=True),
}

VARIANTS_TV = FULL_VARIANTS
EFFDET_COMPILE = False
VARIANTS_EFFDET = [v for v in FULL_VARIANTS
                   if EFFDET_COMPILE or "compile" not in v.name]
print("torchvision :", [v.name for v in VARIANTS_TV])
print("effdet      :", [v.name for v in VARIANTS_EFFDET])

In [ ]:
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")
config = RunConfig(n_warmup=N_WARMUP, n_measure=N_MEASURE, n_profile=N_PROFILE,
                   do_profile=DO_PROFILE, device=DEVICE,
                   compile_backend=CAPS.compile_backend,
                   trt_available=CAPS.flags["tensorrt"], do_int8=DO_INT8)
runner = OptimizationRunner(profile_data, eval_data, coco_gt, config,
                            f"results/fullmodel/{RUN_ID}")
print("Sorties →", runner.run_dir)

## 5. Exécution — un modèle par cellule

Sauvegarde après chaque variante et chaque modèle (survit à une déconnexion).

In [ ]:
runner.run_model(SPECS["retinanet_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["fcos_r50"], VARIANTS_TV)

In [ ]:
runner.run_model(SPECS["efficientdet_d4"], VARIANTS_EFFDET)

## 6. Résultats (sauvegardés)

In [ ]:
df = runner.to_dataframe()
print("Statuts :", dict(df["status"].value_counts()))
df

In [ ]:
spd = runner.speedup_table()
spd.to_csv(runner.run_dir / "speedup.csv", index=False)
spd

In [ ]:
# Échecs (attendus pour certaines combos) — pointeur vers les tracebacks
fails = df[df["status"]=="FAILED"][["model","variant","error"]]
print("Tracebacks :", runner.run_dir/"errors")
fails if len(fails) else "Aucun échec."


## 7. Tout est sauvegardé
```
<préfixe>/results/fullmodel/<RUN_ID>/
  run.log  results.csv  results_final.csv  speedup.csv
  bench/  eval/  modules/  profiles/  errors/
```

In [ ]:
for sub in ["bench","eval","modules","profiles","errors"]:
    d = runner.run_dir/sub
    print(f"  {sub:9s}: {len(list(d.glob('*'))) if d.exists() else 0} fichiers")
print("Artefacts :", runner.run_dir)